# Norm-supervised highway experiments on Colab

This notebook reproduces the experiment matrix from `scripts/run_experiments.sh`. It clones a selected Git ref, creates an isolated virtual environment, runs each configuration through `scripts/test_highway.py`, and writes CSV results directly to Google Drive.

If a model is only a host symlink in git (e.g. `models/3_lanes_30_vehicles.zip`), copy the real zip to Drive and point `DRIVE_MODEL_OVERRIDES` at it. Use a distinct `RUN_NAME` when running multiple Colab sessions in parallel.

In [2]:
# Mount Google Drive. Colab will prompt you to authorize access.
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
# Experiment parameters (edit this cell before running).
from datetime import datetime, timezone

REPO_URL = "https://github.com/thowell332/state-wise-constrained-policy-shaping.git"  #@param {type:"string"}
REPO_REF = "thowell332/results-galore"  #@param {type:"string"}

NUM_EPISODES = 1000  #@param {type:"integer"}
BASE_SEED = 42  #@param {type:"integer"}
ENVIRONMENTS = ["3L30V"]  # Valid: 2L10V, 3L30V, 4L40V, 4L20V, 6L50V
FORCE_WRITE = True  #@param {type:"boolean"}
FILE_PREFIX = "3L30V"  # Preserves the naming used by run_experiments.sh

# Copy real model zips from Drive into the cloned repo when git only has a broken symlink.
# Keys are paths relative to the project root.
DRIVE_MODEL_OVERRIDES = {
    "models/3_lanes_30_vehicles.zip": "/content/drive/MyDrive/aaai2027/models/3_lanes_30_vehicles.zip",
}

# Each tuple is: (profile, method, value, enforce_filter).
# value must be None except for adaptive/fixed methods.
EXPERIMENTS = [
    #("right_lane", "nop", None, False),
    #("right_lane", "nop", None, True),
    #("right_lane", "naive", None, False),
    #("right_lane", "naive", None, True),
    #("right_lane", "adaptive", "0.05", False),
    ("right_lane", "adaptive", "0.05", True),
    #("right_lane", "fixed", "1.00", False),
    #("right_lane", "fixed", "1.00", True),
    #("right_lane", "projection", None, False),
    #("right_lane", "projection", None, True),
    #("right_lane", "adaptive", "0.0316", True),
    #("right_lane", "adaptive", "0.3162", True),
    #("right_lane", "adaptive", "3.1623", True),
    #("right_lane", "adaptive", "10.000", True),
]

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")  #@param {type:"string"}
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/aaai2027/state-wise-constrained-policy-shaping"  #@param {type:"string"}

In [4]:
# Clone the requested ref and create the virtual environment.
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/state-wise-constrained-policy-shaping")
VENV_DIR = Path("/content/venvs/state-wise-constrained-policy-shaping")
ENV_MODEL_FILES = {
    "2L10V": "models/4_lanes_20_vehicles.zip",
    "3L30V": "models/3_lanes_30_vehicles.zip",
    "4L40V": "models/3_lanes_30_vehicles.zip",
    "4L20V": "models/4_lanes_20_vehicles.zip",
    "6L50V": "models/4_lanes_20_vehicles.zip",
}

# Headless Colab rendering for pygame / highway-env.
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OFFSCREEN_RENDERING"] = "1"

for path in (PROJECT_DIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)

# Replace broken git symlinks (or missing models) with real zips from Drive.
for relative_model, drive_src in DRIVE_MODEL_OVERRIDES.items():
    dst = PROJECT_DIR / relative_model
    src = Path(drive_src).expanduser()
    if not src.is_file() or src.stat().st_size == 0:
        raise FileNotFoundError(
            f"Drive model override not found or empty: {src}\n"
            f"Copy the real zip to Drive and update DRIVE_MODEL_OVERRIDES."
        )
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)
    print(f"Installed model override: {dst} <- {src} ({dst.stat().st_size:,} bytes)")


def create_venv(venv_dir: Path) -> Path:
    """Create a Colab-friendly venv that reuses the system site packages (incl. CUDA torch)."""
    venv_dir.parent.mkdir(parents=True, exist_ok=True)
    # Colab images often ship without ensurepip / python3-venv.
    subprocess.run(
        ["apt-get", "install", "-y", "python3-venv", f"python{sys.version_info.major}.{sys.version_info.minor}-venv"],
        check=False,
    )
    result = subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(venv_dir)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("stdlib venv failed; falling back to virtualenv")
        print(result.stderr)
        if venv_dir.exists():
            shutil.rmtree(venv_dir)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
        subprocess.run(
            [sys.executable, "-m", "virtualenv", "--system-site-packages", str(venv_dir)],
            check=True,
        )
    return venv_dir / "bin" / "python"


VENV_PYTHON = create_venv(VENV_DIR)

# Prefer vendored forks over any PyPI / system copies of the same package names.
os.environ["PYTHONPATH"] = str(PROJECT_DIR) + (
    os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
)

subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
# Uninstall PyPI packages that would shadow the in-repo forks (DQN_ME, custom envs).
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "uninstall", "-y", "stable-baselines3", "highway-env"],
    check=False,
)
# Install runtime deps without pulling those shadowed packages.
subprocess.run(
    [
        str(VENV_PYTHON), "-m", "pip", "install",
        "gymnasium>=1.0.0", "numpy", "pandas", "scipy", "matplotlib", "tensorboard",
    ],
    check=True,
)
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", "--no-deps", "-e", str(PROJECT_DIR)],
    check=True,
)

for required in (
    PROJECT_DIR / "scripts/test_highway.py",
    PROJECT_DIR / "stable_baselines3" / "dqn_ME",
    PROJECT_DIR / "highway_env",
    PROJECT_DIR / "supervisor",
):
    if not required.exists():
        raise FileNotFoundError(
            f"Selected Git ref is missing required path: {required}\n"
            "Push the vendored stable_baselines3/highway_env packages to the branch you clone."
        )

# Sanity-check that DQN_ME resolves from the vendored package.
probe = subprocess.run(
    [
        str(VENV_PYTHON), "-c",
        "import stable_baselines3, inspect; "
        "from stable_baselines3 import DQN_ME; "
        "print('stable_baselines3:', stable_baselines3.__file__); "
        "print('DQN_ME:', inspect.getfile(DQN_ME))",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(probe.stdout)

# Fail early if a selected env's model is still missing after Drive overrides.
missing_models = []
for env_name in ENVIRONMENTS:
    if env_name not in ENV_MODEL_FILES:
        raise ValueError(f"Unknown environment {env_name!r}. Valid: {sorted(ENV_MODEL_FILES)}")
    model_path = PROJECT_DIR / ENV_MODEL_FILES[env_name]
    if model_path.is_symlink() and not model_path.exists():
        missing_models.append(f"{model_path} (broken symlink)")
    elif not model_path.is_file() or model_path.stat().st_size == 0:
        missing_models.append(str(model_path))
if missing_models:
    raise FileNotFoundError(
        "Missing model files after clone/overrides. Add them to DRIVE_MODEL_OVERRIDES:\n  - "
        + "\n  - ".join(missing_models)
    )

print(f"Project: {PROJECT_DIR}")
print(f"Python:  {VENV_PYTHON}")
print(f"PYTHONPATH includes: {PROJECT_DIR}")

Installed model override: /content/state-wise-constrained-policy-shaping/models/3_lanes_30_vehicles.zip <- /content/drive/MyDrive/aaai2027/models/3_lanes_30_vehicles.zip (1,209,603 bytes)
stdlib venv failed; falling back to virtualenv
Error: Command '['/content/venvs/state-wise-constrained-policy-shaping/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.

stable_baselines3: /content/state-wise-constrained-policy-shaping/stable_baselines3/__init__.py
DQN_ME: /content/state-wise-constrained-policy-shaping/stable_baselines3/dqn_ME/dqn_ME.py

Project: /content/state-wise-constrained-policy-shaping
Python:  /content/venvs/state-wise-constrained-policy-shaping/bin/python
PYTHONPATH includes: /content/state-wise-constrained-policy-shaping


In [5]:
# Run the experiment matrix. CSV files and a complete log are written to Drive.
import json
import os

RUN_DIR = Path(DRIVE_RESULTS_ROOT).expanduser() / RUN_NAME
RESULTS_DIR = RUN_DIR / "results"
RUN_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "num_episodes": NUM_EPISODES,
    "base_seed": BASE_SEED,
    "environments": ENVIRONMENTS,
    "force_write": FORCE_WRITE,
    "file_prefix": FILE_PREFIX,
    "drive_model_overrides": DRIVE_MODEL_OVERRIDES,
    "experiments": EXPERIMENTS,
}
(RUN_DIR / "run_parameters.json").write_text(json.dumps(manifest, indent=2))

def run_and_tee(command, log_handle):
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONPATH": str(PROJECT_DIR)},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_handle.write(line)
        log_handle.flush()
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

with (RUN_DIR / "run.log").open("a", buffering=1) as log:
    for env_name in ENVIRONMENTS:
        for profile, method, value, filter_enabled in EXPERIMENTS:
            suffix = f"{method}_{'filtered' if filter_enabled else 'unfiltered'}"
            output_dir = RESULTS_DIR / env_name / profile / suffix
            output_dir.mkdir(parents=True, exist_ok=True)
            value_suffix = f"_{value}" if method in {"adaptive", "fixed"} else ""
            output_path = output_dir / f"{FILE_PREFIX}_{env_name}{value_suffix}.csv"

            if output_path.exists() and not FORCE_WRITE:
                message = f"Skipping existing result: {output_path}\n"
                print(message, end="")
                log.write(message)
                continue

            command = [
                str(VENV_PYTHON), str(PROJECT_DIR / "scripts/test_highway.py"),
                "--profile", profile,
                "--method", method,
                "--episodes", str(NUM_EPISODES),
                "--seed", str(BASE_SEED),
                "--env", env_name,
                "--output", str(output_path),
            ]
            if value is not None:
                command += ["--value", str(value)]
            if filter_enabled:
                command.append("--filter")
            run_and_tee(command, log)

print(f"Results saved to: {RUN_DIR}")

$ /content/venvs/state-wise-constrained-policy-shaping/bin/python /content/state-wise-constrained-policy-shaping/scripts/test_highway.py --profile right_lane --method adaptive --episodes 1000 --seed 42 --env 3L30V --output /content/drive/MyDrive/aaai2027/state-wise-constrained-policy-shaping/run_20260722T003850Z/results/3L30V/right_lane/adaptive_filtered/3L30V_3L30V_0.05.csv --value 0.05 --filter
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
2026-07-22 00:39:33.085599: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-22 00:39:33.197494: I tensorflow/core/platform/cpu_feature_guard.cc:210] This Ten

In [7]:
# Analyze results: write markdown under RUN_DIR/analysis and print the summary tables here.
ANALYSIS_DIR = RUN_DIR / "analysis"
RESULTS_DIR = RUN_DIR / "results"

analyze_script = PROJECT_DIR / "scripts" / "analyze_results.py"
if not analyze_script.is_file():
    raise FileNotFoundError(f"Analysis script not found: {analyze_script}")

command = [
    str(VENV_PYTHON),
    str(analyze_script),
    "--results-dir", str(RESULTS_DIR),
    "--output-dir", str(ANALYSIS_DIR),
]
print("$", " ".join(command))
process = subprocess.run(
    command,
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
    env={
        **os.environ,
        "PYTHONUNBUFFERED": "1",
        "PYTHONPATH": str(PROJECT_DIR),
        # Avoid the script's pgf/LaTeX backend on Colab when only generating tables.
        "MPLBACKEND": "Agg",
    },
)
print(process.stdout, end="")
if process.returncode:
    print(process.stderr)
    raise subprocess.CalledProcessError(process.returncode, command)

summary_file = ANALYSIS_DIR / "summary.md"
details_file = ANALYSIS_DIR / "details.md"
if not summary_file.is_file():
    raise FileNotFoundError(f"Expected summary markdown was not written: {summary_file}")

print("\n" + "=" * 72)
print(f"SUMMARY TABLES ({summary_file})")
print("=" * 72 + "\n")
print(summary_file.read_text())

if details_file.is_file():
    print("\n" + "=" * 72)
    print(f"DETAILS TABLES ({details_file})")
    print("=" * 72 + "\n")
    print(details_file.read_text())


$ /content/venvs/state-wise-constrained-policy-shaping/bin/python /content/state-wise-constrained-policy-shaping/scripts/analyze_results.py --results-dir /content/drive/MyDrive/aaai2027/state-wise-constrained-policy-shaping/run_20260722T003850Z/results --output-dir /content/drive/MyDrive/aaai2027/state-wise-constrained-policy-shaping/run_20260722T003850Z/analysis
Scanning for CSV files in /content/drive/MyDrive/aaai2027/state-wise-constrained-policy-shaping/run_20260722T003850Z/results...
Found 1 configuration groups
Generating markdown tables...
Generating summary table...
Analysis complete! Results written to:
  Summary: /content/drive/MyDrive/aaai2027/state-wise-constrained-policy-shaping/run_20260722T003850Z/analysis/summary.md
  Details: /content/drive/MyDrive/aaai2027/state-wise-constrained-policy-shaping/run_20260722T003850Z/analysis/details.md

SUMMARY TABLES (/content/drive/MyDrive/aaai2027/state-wise-constrained-policy-shaping/run_20260722T003850Z/analysis/summary.md)

# Expe

In [6]:
# Verify the artifacts persisted to Drive.
for artifact in sorted(path for path in RUN_DIR.rglob("*") if path.is_file()):
    print(f"{artifact.relative_to(RUN_DIR)}  ({artifact.stat().st_size:,} bytes)")

results/3L30V/right_lane/adaptive_filtered/3L30V_3L30V_0.05.csv  (134,342 bytes)
run.log  (7,991 bytes)
run_parameters.json  (499 bytes)
